# NYC Yellow Taxi — Exploratory Data Analysis

**Dataset:** NYC TLC Yellow Taxi Trip Records — January 2023  
**Source:** https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page  

## Objectives
1. Load and inspect the raw dataset
2. Clean the data (nulls, wrong types, invalid values, duplicates)
3. Explore patterns through aggregations and a merge with zone lookup data
4. Visualize key insights

## 1. Setup

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid')

RAW_DATA_PATH = '../data/raw/yellow_tripdata_2023-01.parquet'
PROCESSED_DATA_PATH = '../data/processed/yellow_tripdata_2023-01_clean.parquet'

## 2. Load Data

In [3]:
df = pd.read_parquet(RAW_DATA_PATH)
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

Shape: 3,066,766 rows × 19 columns


## 3. Initial Inspection

In [4]:
# First rows
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.00,0.97,1.00,N,161,141,2,9.30,1.00,0.50,0.00,0.00,1.00,14.30,2.50,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.00,1.10,1.00,N,43,237,1,7.90,1.00,0.50,4.00,0.00,1.00,16.90,2.50,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.00,2.51,1.00,N,48,238,1,14.90,1.00,0.50,15.00,0.00,1.00,34.90,2.50,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.00,1.90,1.00,N,138,7,1,12.10,7.25,0.50,0.00,0.00,1.00,20.85,0.00,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.00,1.43,1.00,N,107,79,1,11.40,1.00,0.50,3.28,0.00,1.00,19.68,2.50,0.00


In [5]:
# Column names, types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3066766 entries, 0 to 3066765
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int64         
 8   DOLocationID           int64         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  airport_fee            float64   

In [6]:
# Summary statistics for numeric columns
df.describe()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
count,3066766.00,3066766,3066766,2995023.00,3066766.00,2995023.00,3066766.00,3066766.00,3066766.00,3066766.00,3066766.00,3066766.00,3066766.00,3066766.00,3066766.00,3066766.00,2995023.00,2995023.00
mean,1.73,2023-01-17 00:22:26.288164,2023-01-17 00:38:06.427874,1.36,3.85,1.50,166.40,164.39,1.19,18.37,1.54,0.49,3.37,0.52,0.98,27.02,2.27,0.11
min,1.00,2008-12-31 23:01:42,2009-01-01 14:29:11,0.00,0.00,1.00,1.00,1.00,0.00,-900.00,-7.50,-0.50,-96.22,-65.00,-1.00,-751.00,-2.50,-1.25
25%,1.00,2023-01-09 16:21:57.250000,2023-01-09 16:37:06,1.00,1.06,1.00,132.00,114.00,1.00,8.60,0.00,0.50,1.00,0.00,1.00,15.40,2.50,0.00
50%,2.00,2023-01-17 08:42:29.500000,2023-01-17 08:58:30.500000,1.00,1.80,1.00,162.00,162.00,1.00,12.80,1.00,0.50,2.72,0.00,1.00,20.16,2.50,0.00
75%,2.00,2023-01-24 16:26:27,2023-01-24 16:42:49,1.00,3.33,1.00,234.00,234.00,1.00,20.50,2.50,0.50,4.20,0.00,1.00,28.70,2.50,0.00
max,2.00,2023-02-01 00:56:53,2023-02-02 09:28:47,9.00,258928.15,99.00,265.00,265.00,4.00,1160.10,12.50,53.16,380.80,196.99,1.00,1169.40,2.50,1.25
std,0.44,NaN,NaN,0.90,249.58,6.47,64.24,69.94,0.53,17.81,1.79,0.10,3.83,2.02,0.18,22.16,0.77,0.36


In [7]:
# Null count and percentage per column
null_summary = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
null_summary[null_summary['null_count'] > 0]

,null_count,null_pct
passenger_count,71743,2.34
RatecodeID,71743,2.34
store_and_fwd_flag,71743,2.34
congestion_surcharge,71743,2.34
airport_fee,71743,2.34


In [8]:
df[df['passenger_count'].isnull()]['VendorID'].value_counts()


VendorID
2    47086
1    24657
Name: count, dtype: int64

In [9]:
# Duplicate rows
print(f'Duplicate rows: {df.duplicated().sum():,}')

Duplicate rows: 0


## 4. Data Cleaning

Issues identified during inspection:
- ~2.3% of rows have nulls in `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, `airport_fee`
- Pickup dates outside January 2023 (some from 2008)
- `trip_distance` = 0 or unrealistically large values (max seen: 258,928 miles)
- Negative `fare_amount` and `total_amount`
- `passenger_count` = 0

In [10]:
df_clean = df.copy()

initial_rows = len(df_clean)

# Drop rows with nulls in key columns
df_clean = df_clean.dropna(subset=['passenger_count', 'RatecodeID', 'congestion_surcharge', 'airport_fee'])

# Keep only trips within January 2023
df_clean = df_clean[
    (df_clean['tpep_pickup_datetime'] >= '2023-01-01') &
    (df_clean['tpep_pickup_datetime'] < '2023-02-01')
]

# Remove trips with zero or negative distance (max 100 miles)
df_clean = df_clean[
    (df_clean['trip_distance'] > 0) &
    (df_clean['trip_distance'] <= 100)
]

# Remove trips with zero or negative fares
df_clean = df_clean[
    (df_clean['fare_amount'] > 0) &
    (df_clean['total_amount'] > 0)
]

# Remove trips with zero passengers
df_clean = df_clean[df_clean['passenger_count'] > 0]

removed = initial_rows - len(df_clean)
print(f'Rows before cleaning : {initial_rows:>10,}')
print(f'Rows removed         : {removed:>10,} ({removed/initial_rows*100:.1f}%)')
print(f'Rows after cleaning  : {len(df_clean):>10,}')

Rows before cleaning :  3,066,766
Rows removed         :    182,616 (6.0%)
Rows after cleaning  :  2,884,150


In [11]:
# Add trip duration in minutes — derived column useful for analysis
df_clean = df_clean.copy()
df_clean['trip_duration_min'] = (
    df_clean['tpep_dropoff_datetime'] - df_clean['tpep_pickup_datetime']
).dt.total_seconds() / 60

# Keep only sensible durations (1 min to 3 hours)
df_clean = df_clean[
    (df_clean['trip_duration_min'] >= 1) &
    (df_clean['trip_duration_min'] <= 180)
]

print(f'Final row count: {len(df_clean):,}')
df_clean[['trip_distance', 'fare_amount', 'total_amount', 'trip_duration_min']].describe()

Final row count: 2,872,885


,trip_distance,fare_amount,total_amount,trip_duration_min
count,2872885.00,2872885.00,2872885.00,2872885.00
mean,3.43,18.51,27.33,14.55
std,4.42,16.92,21.27,10.98
min,0.01,0.01,1.01,1.00
25%,1.10,8.60,15.48,7.23
50%,1.80,12.80,20.16,11.57
75%,3.37,19.80,28.56,18.28
max,96.70,598.70,614.45,179.60


In [12]:
# Save clean dataset
df_clean.to_parquet(PROCESSED_DATA_PATH, index=False)
print(f'Clean dataset saved to {PROCESSED_DATA_PATH}')

Clean dataset saved to ../data/processed/yellow_tripdata_2023-01_clean.parquet


## 5. Aggregations

Exploring patterns by grouping trips across time and payment type.

In [13]:
# Extract hour of day from pickup datetime
df_clean['pickup_hour'] = df_clean['tpep_pickup_datetime'].dt.hour

# Trips and average fare by hour
by_hour = df_clean.groupby('pickup_hour').agg(
    trip_count=('VendorID', 'count'),
    avg_fare=('fare_amount', 'mean'),
    avg_duration_min=('trip_duration_min', 'mean')
).round(2)

by_hour

,trip_count,avg_fare,avg_duration_min
pickup_hour,,,
0,79129,19.77,13.49
1,55239,17.81,12.51
2,38463,16.70,11.91
3,24847,17.69,11.98
4,15643,22.17,13.37
5,16001,26.37,14.68
6,39744,22.10,14.48
7,79250,18.88,14.37
8,107492,17.39,14.22


In [14]:
# Payment type breakdown
# 1=Credit card, 2=Cash, 3=No charge, 4=Dispute
payment_labels = {1: 'Credit card', 2: 'Cash', 3: 'No charge', 4: 'Dispute'}

by_payment = df_clean.groupby('payment_type').agg(
    trip_count=('VendorID', 'count'),
    avg_fare=('fare_amount', 'mean'),
    avg_tip=('tip_amount', 'mean')
).round(2)

by_payment.index = by_payment.index.map(payment_labels)
by_payment['trip_pct'] = (by_payment['trip_count'] / by_payment['trip_count'].sum() * 100).round(1)
by_payment

,trip_count,avg_fare,avg_tip,trip_pct
payment_type,,,,
Credit card,2345432,18.50,4.17,81.60
Cash,503819,18.53,0.00,17.50
No charge,8292,17.75,0.00,0.30
Dispute,15342,18.90,0.02,0.50


In [15]:
# Top 10 busiest pickup locations
top_pickup_zones = (
    df_clean.groupby('PULocationID')
    .size()
    .sort_values(ascending=False)
    .head(10)
    .rename('trip_count')
    .reset_index()
)

top_pickup_zones

,PULocationID,trip_count
0,132,150728
1,237,140755
2,236,130519
3,161,128755
4,186,104593
5,162,100536
6,142,94705
7,230,93892
8,138,86454
9,170,83762


## 6. Merge with Zone Lookup

The trip dataset only contains numeric location IDs. We join with the zone lookup table to get human-readable names for pickup and dropoff locations.

In [16]:
ZONE_LOOKUP_PATH = '../data/raw/taxi_zone_lookup.csv'

zones = pd.read_csv(ZONE_LOOKUP_PATH)
print(f'Zones shape: {zones.shape}')
zones.head()

Zones shape: (265, 4)


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [17]:
# Merge pickup zone names
df_merged = df_clean.merge(
    zones[['LocationID', 'Zone', 'Borough']].rename(columns={
        'LocationID': 'PULocationID',
        'Zone': 'pickup_zone',
        'Borough': 'pickup_borough'
    }),
    on='PULocationID',
    how='left'
)

# Merge dropoff zone names
df_merged = df_merged.merge(
    zones[['LocationID', 'Zone', 'Borough']].rename(columns={
        'LocationID': 'DOLocationID',
        'Zone': 'dropoff_zone',
        'Borough': 'dropoff_borough'
    }),
    on='DOLocationID',
    how='left'
)

print(f'Shape after merge: {df_merged.shape}')
df_merged[['PULocationID', 'pickup_zone', 'pickup_borough', 'DOLocationID', 'dropoff_zone', 'dropoff_borough']].head()

Shape after merge: (2872885, 25)


,PULocationID,pickup_zone,pickup_borough,DOLocationID,dropoff_zone,dropoff_borough
0,161,Midtown Center,Manhattan,141,Lenox Hill West,Manhattan
1,43,Central Park,Manhattan,237,Upper East Side South,Manhattan
2,48,Clinton East,Manhattan,238,Upper West Side North,Manhattan
3,107,Gramercy,Manhattan,79,East Village,Manhattan
4,161,Midtown Center,Manhattan,137,Kips Bay,Manhattan


In [18]:
# Top 10 busiest pickup zones with names
top_zones = (
    df_merged.groupby(['PULocationID', 'pickup_zone', 'pickup_borough'])
    .size()
    .sort_values(ascending=False)
    .head(10)
    .rename('trip_count')
    .reset_index()
)

top_zones

,PULocationID,pickup_zone,pickup_borough,trip_count
0,132,JFK Airport,Queens,150728
1,237,Upper East Side South,Manhattan,140755
2,236,Upper East Side North,Manhattan,130519
3,161,Midtown Center,Manhattan,128755
4,186,Penn Station/Madison Sq West,Manhattan,104593
5,162,Midtown East,Manhattan,100536
6,142,Lincoln Square East,Manhattan,94705
7,230,Times Sq/Theatre District,Manhattan,93892
8,138,LaGuardia Airport,Queens,86454
9,170,Murray Hill,Manhattan,83762
